In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap, MiniMap
from pathlib import Path
import osmnx as ox
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

DATA_PROC = Path("../../data/processed")
DATA_EXT  = Path("../../data/external")
MAPS_OUT  = Path("../outputs/maps")
MAPS_OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
tracts = gpd.read_file(DATA_PROC / "tract_features_predicted.geojson")
tracts = tracts.to_crs("EPSG:4326")  # Folium needs WGS84

stations  = gpd.read_file(DATA_PROC / "all_stations.geojson").to_crs("EPSG:4326")
hospitals = gpd.read_file(DATA_PROC / "hospitals.geojson").to_crs("EPSG:4326")
hydrants  = gpd.read_file(DATA_PROC / "Fire_Hydrants.geojson").to_crs("EPSG:4326")

# Load road graph for routing
G = ox.load_graphml(DATA_EXT / "road_network.graphml")

print(f"Tracts: {len(tracts)}  |  Stations: {len(stations)}  |  Hospitals: {len(hospitals)}")

Tracts: 2468  |  Stations: 412  |  Hospitals: 165


In [ ]:
# from pyproj import Transformer
# import random

# to_3310 = Transformer.from_crs("EPSG:4326", "EPSG:3310", always_xy=True)
# to_4326 = Transformer.from_crs("EPSG:3310", "EPSG:4326", always_xy=True)

# def find_fastest_firestation_route(tracts_gdf, stations_gdf, G, seed=None):
#     """
#     Pick a random High-risk tract as the fire source, then find which fire station
#     can reach it fastest (shortest road-network distance) and return that route.

#     Returns
#     -------
#     fire_latlon       : (lat, lon) of the fire source
#     station_latlon    : (lat, lon) of the responding fire station
#     route             : list of OSMnx node IDs (station → fire)
#     station_name      : name string for the responding station
#     road_distance_km  : shortest road distance in kilometres
#     """
#     # 1. Random fire source from High-risk tracts
#     high_risk = tracts_gdf[tracts_gdf["predicted_tier_label"] == "High"]
#     if high_risk.empty:
#         high_risk = tracts_gdf
#     fire_row = high_risk.sample(1, random_state=seed).iloc[0]
#     centroid = fire_row.geometry.centroid
#     fire_latlon = (centroid.y, centroid.x)

#     # 2. Snap fire location to road network
#     fire_x, fire_y = to_3310.transform(fire_latlon[1], fire_latlon[0])
#     fire_node = ox.distance.nearest_nodes(G, X=fire_x, Y=fire_y)

#     # 3. Snap ALL stations to road network in one bulk call
#     st = stations_gdf.copy()
#     st_xs, st_ys = zip(*[
#         to_3310.transform(row.geometry.x, row.geometry.y)
#         for _, row in st.iterrows()
#     ])
#     station_nodes = ox.distance.nearest_nodes(G, X=list(st_xs), Y=list(st_ys))

#     # 4. Single-source Dijkstra FROM the fire node (reverse edges = travel TO fire)
#     #    We reverse the graph so distances represent travel *to* fire_node.
#     G_rev = G.reverse(copy=False)
#     dist_to_fire = nx.single_source_dijkstra_path_length(G_rev, fire_node, weight="length")

#     # 5. Pick station with minimum road distance to fire
#     best_dist = float("inf")
#     best_idx  = None
#     for i, snode in enumerate(station_nodes):
#         d = dist_to_fire.get(snode, float("inf"))
#         if d < best_dist:
#             best_dist = d
#             best_idx  = i

#     if best_idx is None:
#         raise ValueError("No reachable fire station found from this fire location.")

#     best_station_row  = st.iloc[best_idx]
#     station_latlon    = (best_station_row.geometry.y, best_station_row.geometry.x)
#     station_node      = station_nodes[best_idx]
#     station_name      = best_station_row.get("STATION_NAME", "Fire Station")

#     # 6. Recover the actual path (station → fire) on the original graph
#     route = nx.shortest_path(G, station_node, fire_node, weight="length")

#     return fire_latlon, station_latlon, route, station_name, best_dist / 1000


# fire_latlon, station_latlon, route, station_name, dist_km = find_fastest_firestation_route(
#     tracts, stations, G, seed=42
# )

# print(f"Fire source  : {fire_latlon}")
# print(f"Responding station : {station_name}  @ {station_latlon}")
# print(f"Road distance: {dist_km:.2f} km  |  Route nodes: {len(route)}")


In [ ]:
# def node_latlon(node_id):
#     lon, lat = to_4326.transform(G.nodes[node_id]["x"], G.nodes[node_id]["y"])
#     return (lat, lon)

# route_coords = [node_latlon(n) for n in route]
# print(f"Route has {len(route_coords)} waypoints along the road network.")


In [ ]:
# mid_lat = (fire_latlon[0] + station_latlon[0]) / 2
# mid_lon = (fire_latlon[1] + station_latlon[1]) / 2

# route_map = folium.Map(location=[mid_lat, mid_lon], zoom_start=12, tiles="CartoDB positron")

# # Risk choropleth background
# for _, row in tracts.iterrows():
#     folium.GeoJson(
#         row["geometry"].__geo_interface__,
#         style_function=lambda feat, color=tier_color(row["predicted_tier_label"]): {
#             "fillColor": color, "color": "white",
#             "weight": 0.3, "fillOpacity": 0.45,
#         }
#     ).add_to(route_map)

# # Optimal route: fire station → fire source
# folium.PolyLine(
#     route_coords,
#     color="#e74c3c",
#     weight=5,
#     opacity=0.9,
#     tooltip=f"Fastest route — {dist_km:.2f} km"
# ).add_to(route_map)

# # Fire source marker
# folium.Marker(
#     fire_latlon,
#     icon=folium.Icon(color="red", icon="fire", prefix="fa"),
#     tooltip=f"Fire Source (High-Risk Tract: {fire_latlon[0]:.4f}, {fire_latlon[1]:.4f})"
# ).add_to(route_map)

# # Responding fire station marker
# folium.Marker(
#     station_latlon,
#     icon=folium.Icon(color="blue", icon="truck", prefix="fa"),
#     tooltip=f"Responding Station: {station_name}"
# ).add_to(route_map)

# MiniMap().add_to(route_map)

# route_map.save(MAPS_OUT / "03_evacuation_route.html")
# print(f"Saved 03_evacuation_route.html")
# print(f"Station '{station_name}' → fire in {dist_km:.2f} km via road network")
